# Panorama inference

Loads the trained SegFormer and runs tiled inference on a full panorama, then renders the intergrowth overlay and prints the ore-sort verdict.

In [ ]:
import sys, pathlib, cv2, torch, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
sys.path.insert(0, str(pathlib.Path.cwd() / 'src'))
from orenet import constants as C
from orenet.model import SegmenterConfig, build_segmenter
from orenet.inference import predict
from orenet.grains import extract_grains
from orenet.classify import classify_image
from orenet.viz import make_overlay, colorize_uncertainty

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_segmenter(SegmenterConfig(pretrained=False, n_classes=C.NUM_CLASSES))
model.load_state_dict(torch.load('outputs/segformer_phases.pt', map_location=device, weights_only=True))
model.to(device).eval()
print('loaded model on', device)

In [ ]:
pano_paths = sorted((C.PANORAMS).glob('*.jpg'))
print('panoramas:', [p.name for p in pano_paths])
pano = cv2.imread(str(pano_paths[0]), cv2.IMREAD_COLOR)
# downscale very large panoramas for a quick demo pass
if max(pano.shape[:2]) > 8000:
    scale = 8000 / max(pano.shape[:2])
    pano = cv2.resize(pano, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
print('panorama shape:', pano.shape)

In [ ]:
import time
t0 = time.time()
class_map, uncertainty = predict(model, pano, device, C.NUM_CLASSES, tile=1024, overlap=0.25)
print(f'inference {time.time()-t0:.1f}s, class_map {class_map.shape}')
grains = extract_grains(class_map)
res = classify_image(class_map, grains)
print(f"VERDICT: {res.ore_class} | talc={res.talc_frac*100:.1f}% "
      f"fine={res.fine_frac*100:.1f}% sulfide={res.sulfide_frac*100:.1f}% grains={len(grains)}")

In [ ]:
overlay = make_overlay(pano, class_map, grains)
fig, ax = plt.subplots(1, 3, figsize=(18, 7))
ax[0].imshow(cv2.cvtColor(pano, cv2.COLOR_BGR2RGB)); ax[0].set_title('panorama'); ax[0].axis('off')
ax[1].imshow(overlay); ax[1].set_title('overlay (green/red/blue)'); ax[1].axis('off')
ax[2].imshow(colorize_uncertainty(uncertainty)); ax[2].set_title('uncertainty'); ax[2].axis('off')
plt.tight_layout(); plt.show()
cv2.imwrite('outputs/panorama_overlay.png', cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))